[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/text_features/text_features.ipynb)

# Use the answers as features

A downstream model sometimes wants a number per text, not a chat completion. FRIDA-Decisions turns each review into a few features: three yes/no probabilities and one score. A logistic regression, fit in code on a toy split, then decides which reviews to escalate. Swapping a question changes a column of the matrix. It does not change the regression.

This is a dozen labelled reviews, not a benchmark. The printout includes a majority-class baseline so the regression has something dull to beat.

Pattern: a small version of [Autoresearch feature discovery](https://docs.typesafe.ai/cookbooks/autoresearch_feature_discovery). Labels are synthetic.


In [1]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0" scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


Four numbers per review. The severity score is divided by 4 so it sits on the same 0–1 scale as the yes/no probabilities.


In [3]:
try:
    from sklearn.linear_model import LogisticRegression
except ImportError as error:
    raise SystemExit("pip install scikit-learn") from error

DATA = [
    {"split": "train", "escalate": 1, "text": "Заказ №4412 пришёл разбитым, верните деньги сегодня, я зол."},
    {"split": "train", "escalate": 1, "text": "Требую возврат за наушники, которые хрипят со второго дня, заказ 7781."},
    {"split": "train", "escalate": 1, "text": "Курьер 12 марта не привёз посылку и трубку не брал. Хочу, чтобы это исправили."},
    {"split": "train", "escalate": 1, "text": "Списали дважды за заказ 1904. Немедленно верните лишнее, это уже не первый раз."},
    {"split": "train", "escalate": 0, "text": "Чайник красивый, кипятит быстро, спасибо за упаковку."},
    {"split": "train", "escalate": 0, "text": "Книга как в описании, доставка вовремя."},
    {"split": "train", "escalate": 0, "text": "Нормальные наушники за эти деньги, пользуюсь неделю."},
    {"split": "train", "escalate": 0, "text": "Спасибо, оператор всё объяснил и заказ нашёлся."},
    {"split": "test", "escalate": 1, "text": "Заказ 3308 протекает, верните оплату, я не буду это чинить сам."},
    {"split": "test", "escalate": 1, "text": "Обещали доставку 4 апреля и не привезли. Исправьте или верните деньги."},
    {"split": "test", "escalate": 0, "text": "Лампа яркая, мужу понравилась, претензий нет."},
    {"split": "test", "escalate": 0, "text": "Обычный кабель, работает, как и ждал."},
    {"split": "test", "escalate": 1, "text": "Пылесос дымит с первого включения, заказ 6620, жду замену и извинение."},
    {"split": "test", "escalate": 0, "text": "Понравился сервис, ответили быстро и по делу."},
    {"split": "train", "escalate": 1, "text": "Хватит кормить отписками: заказ 5150 бракованный, оформите возврат."},
    {"split": "test", "escalate": 0, "text": "Чашка целая, цвет как на фото."},
]
FEATURES = ("angry", "demand", "concrete", "severity")
SEVERITY = [
    "Спокойный отзыв, делать ничего не нужно.",
    "Лёгкое недовольство, можно ответить шаблоном.",
    "Есть претензия, стоит прочитать.",
    "Явная претензия, лучше ответить человеку.",
    "Жёсткая претензия, откладывать нельзя.",
]


def request_for(text):
    return {
        "state": text,
        "questions": {
            "angry": {"type": "noul", "instructions": "Автор раздражён?"},
            "demand": {"type": "noul", "instructions": "Автор требует вернуть деньги или что-то исправить?"},
            "concrete": {"type": "noul", "instructions": "Назван конкретный заказ, товар или дата?"},
            "severity": {"type": "score", "instructions": "Насколько отзыв требует реакции?",
                         "criteria": SEVERITY},
        },
    }


def featurize(answers):
    return [
        answers["angry"]["noul"],
        answers["demand"]["noul"],
        answers["concrete"]["noul"],
        answers["severity"]["score"] / 4.0,
    ]


def majority_label(labels):
    best = labels[0]
    best_count = 0
    for label in set(labels):
        count = labels.count(label)
        if count > best_count:
            best = label
            best_count = count
    return best


The regression only sees the four numbers. The split is fixed in `DATA`. The coefficients show which question the linear model leaned on.


In [4]:
def run(judge):
    responses = judge.judge_batch([request_for(row["text"]) for row in DATA])
    train_x = []
    train_y = []
    test_rows = []
    test_x = []
    for row, response in zip(DATA, responses):
        features = featurize(response["answers"])
        if row["split"] == "train":
            train_x.append(features)
            train_y.append(row["escalate"])
        else:
            test_rows.append(row)
            test_x.append(features)
    model = LogisticRegression(max_iter=400)
    model.fit(train_x, train_y)
    predicted = model.predict(test_x)
    hits = 0
    for row, pred in zip(test_rows, predicted):
        if int(pred) == row["escalate"]:
            hits += 1
    majority = majority_label(train_y)
    baseline = 0
    for row in test_rows:
        if row["escalate"] == majority:
            baseline += 1
    coefs = []
    for name, coef in zip(FEATURES, model.coef_[0]):
        coefs.append(f"{name}={coef:.2f}")
    print("features:", ", ".join(FEATURES))
    print("coefficients:", " ".join(coefs))
    print(f"test accuracy {hits}/{len(test_rows)}")
    print(f"majority baseline (label {majority}) {baseline}/{len(test_rows)}")
    for row, pred in zip(test_rows, predicted):
        if int(pred) == row["escalate"]:
            mark = "ok"
        else:
            mark = "miss"
        print(f"  {mark:4} pred {int(pred)} gold {row['escalate']}  {row['text'][:64]}")


In [5]:
run(judge)


features: angry, demand, concrete, severity
coefficients: angry=0.62 demand=0.89 concrete=-0.36 severity=0.99
test accuracy 7/7
majority baseline (label 1) 3/7
  ok   pred 1 gold 1  Заказ 3308 протекает, верните оплату, я не буду это чинить сам.
  ok   pred 1 gold 1  Обещали доставку 4 апреля и не привезли. Исправьте или верните д
  ok   pred 0 gold 0  Лампа яркая, мужу понравилась, претензий нет.
  ok   pred 0 gold 0  Обычный кабель, работает, как и ждал.
  ok   pred 1 gold 1  Пылесос дымит с первого включения, заказ 6620, жду замену и изви
  ok   pred 0 gold 0  Понравился сервис, ответили быстро и по делу.
  ok   pred 0 gold 0  Чашка целая, цвет как на фото.
